In [7]:
import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.tracking import MlflowClient

# Set MLflow tracking URI (local storage)
MLFLOW_TRACKING_URI = "file:./mlruns"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Load the latest version of the registered model
model_name = "FakeNewsModel"
client = MlflowClient()

latest_versions = client.get_latest_versions(model_name, stages=["None", "Production", "Staging"])
if not latest_versions:
    raise ValueError("No versions of the model found in MLflow.")

latest_version = latest_versions[0].version  # Get the most recent version
model_uri = f"models:/{model_name}/{latest_version}"

# Load the model
model = mlflow.pyfunc.load_model(model_uri)
print(f"Loaded model version: {latest_version}")

# Prediction function
def predict_news(text):
    """Runs prediction on input text and logs results in MLflow."""
    
    # Check if there is an active run and end it if necessary
    if mlflow.active_run() is not None:
        mlflow.end_run()  # End the current active run

    # Start a new run
    with mlflow.start_run(run_name="latest_prediction"):
        input_text = pd.Series([text])
        prediction = model.predict(input_text)[0]
        
        # Log input and prediction
        mlflow.log_param("input_text", text)
        mlflow.log_metric("prediction", int(prediction))
        
        print(f"Prediction: {'Real' if prediction == 1 else 'Fake'}")
        return prediction

# Example usage
if __name__ == "__main__":
    sample_text = "Breaking: Scientists discover new renewable energy source"
    predict_news(sample_text)


C:\Users\Doha\AppData\Local\Temp\ipykernel_3524\3727255613.py:14: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.10.0/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(model_name, stages=["None", "Production", "Staging"])
2025/04/05 12:27:05 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 2.21.3, required: mlflow==2.10.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


Loaded model version: 1
Prediction: Fake


In [1]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

# Initialize the MLflow client
client = MlflowClient()

# Model name and desired stage
model_name = "FakeNewsModel"
stage = "Production"

# Try to load the model from the 'Production' stage
try:
    # Search for the model version in the desired stage
    model_versions = client.get_latest_versions(model_name, stages=[stage])

    if model_versions:
        # Load the model from the specified stage
        model_uri = f"models:/{model_name}/{stage}"
        model = mlflow.pyfunc.load_model(model_uri)
        print(f"Loaded model from {model_uri}")
    else:
        raise RuntimeError(f"No model found in stage '{stage}' for {model_name}")

except Exception as e:
    print(f"Error loading model from MLflow: {str(e)}")

# Example input text for prediction
input_text = pd.Series(["Breaking news: Stock prices soar to new heights"])

# If the model is loaded, make the prediction
if 'model' in locals():
    prediction = model.predict(input_text)
    prediction_label = "Real" if prediction[0] == 1 else "Fake"

    # Print out the prediction
    print(f"Prediction: {prediction_label}")


Error loading model from MLflow: No model found in stage 'Production' for FakeNewsModel


C:\Users\Doha\AppData\Local\Temp\ipykernel_9244\414150705.py:15: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  model_versions = client.get_latest_versions(model_name, stages=[stage])


In [5]:
import mlflow.pyfunc

# Define the model URI for the 'Production' stage
model_uri = "models:/FakeNewsModel/Production"

# Load the model
model = mlflow.pyfunc.load_model(model_uri)

# Example input text for prediction
input_text = ["Breaking news: Stock prices soar to new heights"]

# Make the prediction
prediction = model.predict(input_text)

# Print the prediction
print(f"Prediction: {'Real' if prediction[0] == 1 else 'Fake'}")


Prediction: Fake


In [1]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")  # Make sure this is the same everywhere
